In [1]:
import tensorflow as tf

2025-07-02 15:21:18.732175: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 15:21:18.733096: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 15:21:18.737210: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 15:21:18.747035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751469678.762130   85319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751469678.76

In [2]:
latent_size = 64
_DEFAULT_INITIALIZERS = {"w": tf.compat.v1.keras.initializers.VarianceScaling(scale=1.0, mode="fan_avg", distribution="uniform",seed=111),
                         "b": tf.compat.v1.zeros_initializer()}

In [3]:
from migration.models.distributions import NormalApproximatePosterior, ConditionalNormalDistribution

prior = ConditionalNormalDistribution(size=latent_size, hidden_layer_size=latent_size, sigma_min=0, raw_sigma_bias=0.25, initializers=_DEFAULT_INITIALIZERS)
approx_posterior = NormalApproximatePosterior(size=latent_size, hidden_layer_size=latent_size, sigma_min=0, raw_sigma_bias=0.25, initializers=_DEFAULT_INITIALIZERS)

In [4]:
rnn_out = tf.random.uniform((32,64), dtype=tf.float32)
targets_encoded = tf.random.uniform((32,64), dtype=tf.float32)

2025-07-02 15:21:24.698406: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
tf.concat(targets_encoded, axis=1)

<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
array([[0.02754807, 0.6510503 , 0.2209785 , ..., 0.19972527, 0.1057111 ,
        0.89382553],
       [0.06704199, 0.03331387, 0.6578989 , ..., 0.35309446, 0.509673  ,
        0.9515641 ],
       [0.0575881 , 0.5291164 , 0.1472733 , ..., 0.09560907, 0.6961665 ,
        0.16863549],
       ...,
       [0.09005773, 0.4231745 , 0.263039  , ..., 0.34312034, 0.58048105,
        0.65559006],
       [0.0476532 , 0.24971962, 0.5507319 , ..., 0.32422805, 0.54645455,
        0.57390726],
       [0.67015684, 0.16491067, 0.14223278, ..., 0.18002725, 0.37764955,
        0.6242721 ]], dtype=float32)>

In [6]:
latent_dist_prior = prior(rnn_out)

In [7]:
latent_dist_q = approx_posterior(tf.concat([rnn_out, targets_encoded], axis=1), prior_mu=latent_dist_prior.loc)

In [8]:
latent_state = latent_dist_q.sample(seed=1)  
latent_state_prior = latent_dist_prior.sample(seed=1)

In [9]:
latent_state

<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
array([[-1.9028888 ,  1.5519072 ,  0.07387843, ..., -2.0477142 ,
        -1.3054278 ,  0.53498733],
       [-0.20156619, -0.7734741 ,  0.03293895, ..., -1.2251925 ,
        -0.6182236 ,  1.1152959 ],
       [-0.6706078 ,  0.23291232,  0.77380824, ..., -1.7205853 ,
        -0.3873933 , -1.2161052 ],
       ...,
       [ 0.71637654,  1.0868644 , -0.7362273 , ..., -0.8902395 ,
        -0.9483301 , -0.15258747],
       [-1.1190879 ,  0.08197394,  0.30920094, ..., -1.4799609 ,
        -0.3601493 , -0.8834976 ],
       [ 0.03895426,  1.6028874 ,  1.4236524 , ..., -0.6268033 ,
        -1.1220431 ,  1.2121993 ]], dtype=float32)>

In [10]:
latent_state_prior

<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
array([[-0.75156105,  0.14124303,  0.7847905 , ..., -0.09164259,
        -0.8084781 , -0.52322584],
       [-1.5712957 , -1.556386  , -1.089111  , ..., -1.8297215 ,
        -0.958426  ,  0.08255411],
       [-0.6541796 ,  0.35458142,  0.39925042, ...,  0.07629776,
        -0.5312659 , -0.9687762 ],
       ...,
       [ 0.0102832 ,  0.33447704, -0.05733274, ...,  0.14330557,
        -1.0810153 , -0.13473655],
       [-1.763113  ,  2.1023233 , -0.99915284, ...,  0.2595089 ,
        -0.12988202, -1.3007951 ],
       [-0.62296236,  0.54465246, -2.4501612 , ..., -1.2283251 ,
        -2.2702508 , -0.14681643]], dtype=float32)>